# Normal Chain

In [55]:
import os
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv, find_dotenv
import pandas as pd

In [56]:
_ = load_dotenv (find_dotenv ()) # read local .env file
if not os.getenv("ANTHROPIC_API_KEY"):
    raise ValueError("Cannot find ANTHROPIC_API_KEY，Check .env")

In [57]:
df = pd.read_csv('Reviews1.csv')
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [58]:
# Method 1: Print each complete Text
texts = df.loc[df["ProductId"] == "B001E4KFG0", "Text"]
for i, t in enumerate(texts, 1):
    print(f"\n--- Text {i} ---\n{t}")


--- Text 1 ---
I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.


In [59]:
model = ChatAnthropic(
    model="claude-opus-4-5",  
    temperature=0
)

In [60]:
prompt = ChatPromptTemplate.from_template(
    "What is the best sentence to summarize the comments with in 4 words: {Text}?"
)

In [61]:
chain = prompt | model
result = chain.invoke({"Text": "I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most"})
print(result.content)

"Quality dog food recommended"


# Sequential Chain

In [13]:
from langchain_classic.chains.sequential import SequentialChain
from langchain_classic.chains import LLMChain

In [21]:
first_prompt = ChatPromptTemplate.from_template(
    "What is the best sentence to summarize the comments with in 4 words: {Text}?"
)
chain_1 = LLMChain(llm=model, prompt=first_prompt, output_key="summary")

In [22]:
second_prompt = ChatPromptTemplate.from_template(
    "What language is the following sentence in: {Text}?")
chain_2 = LLMChain(llm=model, prompt=second_prompt, output_key="language")

In [23]:
# Prompt template 3: follow up message
third_prompt = ChatPromptTemplate.from_template (
"Write a follow up response to the following "
"summary in the specified language:"
"\n\nSummary: {summary} \n\nLanguage: {language}"
)
chain_3 = LLMChain (llm=model, prompt=third_prompt, output_key="follow_up")

In [25]:
overall_chain = SequentialChain(
chains=[chain_1, chain_2, chain_3], 
input_variables=["Text"], 
output_variables=["summary", "language", "follow_up"],
verbose = True)

In [51]:
review = df.loc[df["ProductId"] == "B001E4KFG0", "Text"].iloc[0]
result = overall_chain.invoke({"Text": review})
for key, value in result.items():
    print(f"{key}:\n{value}\n")



> Entering new SequentialChain chain...

> Finished chain.
Text:
I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.

summary:
"Quality dog food recommended"

language:
The sentence is in **English**.

This is a product review written in standard American English, discussing Vitality brand canned dog food and the reviewer's Labrador's preference for it.

follow_up:
## Follow-up Response

Thank you for your helpful review! It's great to hear that your Labrador enjoys Vitality canned dog food. A few follow-up questions that other pet owners might find useful:

- **How long have you been feeding your dog this brand?** Long-term feedback is always valuable for those considering a switch.

- **Have you noticed any improvements in your dog's coat, energy levels, or digestion** sinc

# Router Chain

#### 1. Define subchains

In [27]:
physics_chain = """
You are a physics tutor. Answer only physics questions.
Rules:
- Use correct physical principles, units, and sign conventions.
- Show key steps and equations briefly, then give the final result.
- If information is missing, state what assumption you make.

Here is a question: {input}
"""

math_chain = """
You are a mathematics tutor. Answer only mathematics questions.
Rules:
- Provide a clear, step-by-step solution.
- Define symbols you introduce.
- Keep the reasoning concise, and end with the final answer clearly stated.

Here is a question: {input}
"""

history_chain = """
You are a history educator. Answer only history questions.
Rules:
- Provide context (who/what/when/where) before analysis.
- Explain causes, consequences, and significance.
- If dates or actors are uncertain, acknowledge uncertainty rather than guessing.

Here is a question: {input}
"""

cs_chain = """
You are a computer science assistant. Answer only computer science questions.
Rules:
- Prefer precise definitions and practical explanations.
- When relevant, include algorithmic complexity (time/space) or trade-offs.
- If code is requested, provide a minimal working example and brief explanation.

Here is a question: {input}
"""

#### 2. Configure the route list (name, description, and the actual execution prompt (based on the sub-chain name)).

In [28]:
# Configure the names and descriptions of the destination chains for the four routers using a single prompt_infos file.
prompt_infos = [
    {
        "name": "physics",
        "description": "Suitable for physics problems: formulas, units, and law derivation.",
        "prompt_template": physics_chain,
    },
    {
        "name": "math",
        "description": "Suitable for math problems involving step-by-step calculations, symbol definitions, and final results.",
        "prompt_template": math_chain,
    },
    {
        "name": "history",
        "description": "Suitable for history questions: background, timeline, causes and effects.",
        "prompt_template": history_chain,
    },
    {
        "name": "computer_science",
        "description": "Suitable for computer science problems: programming, algorithm complexity, and engineering trade-offs.",
        "prompt_template": cs_chain,
    },
]

In [31]:
from langchain_classic.chains.router import MultiPromptChain, LLMRouterChain
from langchain_classic.chains.router.llm_router import RouterOutputParser
from langchain_anthropic import ChatAnthropic
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

model = ChatAnthropic(
    model="claude-opus-4-5",
    temperature=0
)

memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=model,
    memory=memory,
    verbose=False
)

In [33]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=model, prompt=default_prompt)

#### 3. A `desination_chain` is constructed to traverse `prompt_info`, encapsulating each template into an `LLMChain`. This forms a mapping table of "name -> executable subchain".

In [ ]:
# The execution mapping table is used: the key is the chain name, and the value is the child chain corresponding to that name; after the routing results are obtained, the corresponding child chain is found and executed according to this mapping table.
destination_chains = {}

# Iterate through prompt_infos and dynamically create each destination chain according to the configuration
for p_info in prompt_infos:

    # Read Name and Template
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]

    # Compile the template string into a chat prompt template
    prompt = ChatPromptTemplate.from_template(template=prompt_template)

    # Use the same LLM to build subchains
    chain = LLMChain(llm=model, prompt=prompt)

    # Register to the routing destination dictionary
    destination_chains[name] = chain

#### 4. Generate destination_str so that the routing model knows which destinations are available.

In [38]:
# Generate destination description text for the router prompt
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

# Print the final string (this is for the router prompt, letting the router know the name and function description of each destination)
print(destinations_str)

physics: Suitable for physics problems: formulas, units, and law derivation.
math: Suitable for math problems involving step-by-step calculations, symbol definitions, and final results.
history: Suitable for history questions: background, timeline, causes and effects.
computer_science: Suitable for computer science problems: programming, algorithm complexity, and engineering trade-offs.


#### 5. Build output template

In [39]:
# Set up a prompt template for the router, telling the routing model to do three things:
# 1. Read user input
# 2. Decide which chain name to execute
# 3. Output in a fixed JSON format
MULTI_PROMIT_ROUTER_TEMPLETE = """Given a raw text input to a language model, select the model prompt best suited for the input.

You will be given the names of available prompts and a description of what each prompt is best suited for.
You may also revise the original input if needed.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted like:
```json
{
  \"destination\": string,  // one of the prompt names or \"DEFAULT\"
  \"next_inputs\": string   // revised or original input
}
```

REMEMBER: \"destination\" MUST be one of the candidate prompt names specified below OR \"DEFAULT\".
REMEMBER: \"next_inputs\" can be the original input unchanged.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{input}

<< OUTPUT (must be valid JSON in a markdown code block) >>
"""

#### 6. The output template, candidate chains, output parser, etc. are packaged into a prompt template and combined with the LLM to form an executable chain.

In [46]:
from langchain_core.prompts import PromptTemplate

output_parser = RouterOutputParser()

# Escape JSON example braces in the router template, while keeping real placeholders
router_template = (
    MULTI_PROMIT_ROUTER_TEMPLETE
    .replace("{", "{{")
    .replace("}", "}}")
    .replace("{{destinations}}", "{destinations}")
    .replace("{{input}}", "{input}")
)

# router_prompt is used to make the routing model select destination in fixed output format and pass next_inputs
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    partial_variables={
        "destinations": destinations_str
    },
    output_parser=output_parser,
)

router_chain = LLMRouterChain.from_llm(model, router_prompt)


#### 7. Finally assemble the MultiPromptChain

In [47]:
chain = MultiPromptChain(router_chain=router_chain,
   destination_chains=destination_chains, 
   default_chain=default_chain, 
   verbose=True
)

In [49]:
result = chain.invoke({"input": "What is black body radiation?"})
print(result['text'])



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.
# Black Body Radiation

## Definition
**Black body radiation** is the electromagnetic radiation emitted by an idealized object (a "black body") that absorbs all incident radiation and emits radiation solely based on its temperature.

## Key Physical Principles

**1. Planck's Law** - Describes the spectral distribution:
$$B(\nu, T) = \frac{2h\nu^3}{c^2} \cdot \frac{1}{e^{h\nu/k_BT} - 1}$$

**2. Stefan-Boltzmann Law** - Total power radiated per unit area:
$$P = \sigma T^4$$
where σ = 5.67 × 10⁻⁸ W/(m²·K⁴)

**3. Wien's Displacement Law** - Peak wavelength:
$$\lambda_{max} = \frac{b}{T}$$
where b = 2.898 × 10⁻³ m·K

## Key Characteristics
- Emission depends **only on temperature**, not material composition
- Hotter objects emit more radiation and peak at shorter wavelengths
- Examples: stars, heated metals, cosmic microwave background (T ≈ 2.7 K)

## Historical Significance
The 